# 04 To implement an LSTM-based model to predict POS tags for each word in a sentence by learning context and word dependencies.

In [1]:
import numpy as np
import nltk
from nltk.corpus import treebank
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, TimeDistributed
from sklearn.model_selection import train_test_split

In [3]:
nltk.download('treebank')

[nltk_data] Downloading package treebank to
[nltk_data]     C:\Users\kambl\AppData\Roaming\nltk_data...
[nltk_data]   Package treebank is already up-to-date!


True

In [4]:
sentences = treebank.tagged_sents()

# Separate words and tags

In [5]:
words = [[word for word, tag in sent] for sent in sentences]
tags = [[tag for word, tag in sent] for sent in sentences]

# Create vocabulary

In [5]:
word_vocab = {w: i+1 for i, w in enumerate(set(w for sent in words for w in sent))}
tag_vocab = {t: i+1 for i, t in enumerate(set(t for sent in tags for t in sent))}

# Convert to numeric sequences

In [7]:
X = [[word_vocab[w] for w in sent] for sent in words]
y = [[tag_vocab[t] for t in sent] for sent in tags]

# Padding

In [8]:
max_len = 40
X = pad_sequences(X, maxlen=max_len, padding='post')
y = pad_sequences(y, maxlen=max_len, padding='post')

# Train-test split

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Build LSTM model

In [10]:
model = Sequential()
model.add(Embedding(input_dim=len(word_vocab)+1, output_dim=64, input_length=max_len))
model.add(LSTM(64, return_sequences=True))
model.add(TimeDistributed(Dense(len(tag_vocab)+1, activation='softmax')))

C:\Users\kambl\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [11]:
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])


In [12]:
model.fit(X_train, y_train, batch_size=32, epochs=5)


Epoch 1/5
98/98 ━━━━━━━━━━━━━━━━━━━━ 13s 46ms/step - accuracy: 0.3870 - loss: 2.7725
Epoch 2/5
98/98 ━━━━━━━━━━━━━━━━━━━━ 4s 41ms/step - accuracy: 0.5679 - loss: 1.6852
Epoch 3/5
98/98 ━━━━━━━━━━━━━━━━━━━━ 4s 39ms/step - accuracy: 0.7537 - loss: 0.9995
Epoch 4/5
98/98 ━━━━━━━━━━━━━━━━━━━━ 4s 41ms/step - accuracy: 0.8895 - loss: 0.5528
Epoch 5/5
98/98 ━━━━━━━━━━━━━━━━━━━━ 4s 43ms/step - accuracy: 0.9505 - loss: 0.2943


# Evaluate

In [13]:
loss, accuracy = model.evaluate(X_test, y_test)
print("Accuracy:", accuracy)

25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step - accuracy: 0.9355 - loss: 0.2838
Accuracy: 0.9356321692466736
